# GeoPhys in five minutes

GeoPhys passes each frame of a video through a frozen image encoder and looks at the
*trajectory* the frame features trace out. Physically implausible motion bends and breaks
that trajectory. Nothing is trained.

This notebook makes a few toy videos of a bouncing ball, some plausible and some not, and
shows (1) the trajectory geometry frame by frame and (2) GeoPhys scores per video.
A GPU runtime is recommended (Runtime > Change runtime type).

In [ ]:
# install (in Colab); skip if you already have GeoPhys
!git clone -q https://github.com/ChristianInterno/GeoPhys.git
!pip install -q --no-deps -e GeoPhys   # Colab already has torch, transformers, opencv
import sys; sys.path.insert(0, "GeoPhys")   # importable without restarting the runtime

## 1. Toy videos

A ball is thrown and bounces. In the *plausible* videos it follows gravity. In the others
it either **vanishes** for a while and comes back, or **teleports** sideways half-way through.

In [ ]:
import os, numpy as np, cv2

def make_video(path, kind, seed, T=60, S=224):
    rng = np.random.default_rng(seed)
    bg = cv2.resize((rng.random((14, 14, 3)) * 80 + 90).astype(np.uint8), (S, S), interpolation=cv2.INTER_NEAREST)
    x, y = 30.0, 60.0 + rng.uniform(-10, 10)
    vx, vy, g = 2.6 + rng.uniform(-0.4, 0.4), -4.0 + rng.uniform(-0.5, 0.5), 0.25
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 24, (S, S))
    for t in range(T):
        f = bg.copy()
        cv2.rectangle(f, (0, S - 12), (S, S), (60, 60, 60), -1)
        vy += g; x += vx; y += vy
        if y > S - 24:                                   # bounce on the floor
            y, vy = S - 24, -0.8 * vy
        px = x + (60 if kind == "teleport" and t >= 30 else 0)
        if not (kind == "vanish" and 25 <= t < 40):
            cv2.circle(f, (int(px) % S, int(y)), 12, (30, 30, 200), -1)
        out.write(f)
    out.release()
    return path

os.makedirs("toy", exist_ok=True)
videos = {k: [make_video("toy/%s_%d.mp4" % (k, s), k, s) for s in range(6)]
          for k in ("plausible", "vanish", "teleport")}
print({k: len(v) for k, v in videos.items()})

## 2. The trajectory, frame by frame

Each frame becomes one feature vector (DINOv2, block 12, patch tokens averaged). The
*curvature* at a frame is the angle between the step into it and the step out of it.
A smooth, plausible motion gives a low, steady curve; a violation shows up as a spike.

In [ ]:
import torch, matplotlib.pyplot as plt
from geophys.backbones import get_backbone, load_frames, prep_frames
from geophys.extract import _forward

device = "cuda" if torch.cuda.is_available() else "cpu"
enc = get_backbone("dinov2", device)

def curvature(path, layer="12"):
    pooled, _ = _forward(enc, prep_frames(load_frames(path), "direct"), device)
    z = pooled[layer]["mean"].double().cpu().numpy()
    v = np.diff(z, axis=0)
    cos = (v[:-1] * v[1:]).sum(1) / (np.linalg.norm(v[:-1], axis=1) * np.linalg.norm(v[1:], axis=1) + 1e-12)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

plt.figure(figsize=(8, 3))
for k in videos:
    plt.plot(curvature(videos[k][0]), label=k)
plt.xlabel("frame"); plt.ylabel("curvature (degrees)"); plt.legend(); plt.tight_layout()

## 3. One score per video

`geophys.rate` reads five signals off each trajectory (curvature, angle consistency,
speed variation, acceleration, prediction residual), turns each into a percentile among
the videos scored together, and averages them: 0.5 is typical, higher is less plausible.
The default encoder is DINOv3; DINOv2 is used here because it needs no licence step on
Hugging Face.

In [ ]:
import geophys
allv = [v for k in videos for v in videos[k]]
rated = {r["video"]: r["score"] for r in geophys.rate(allv, backbone="dinov2", device=device)}
for k in videos:
    s = [rated[os.path.abspath(v)] for v in videos[k]]
    print("%-10s mean score %.3f" % (k, np.mean(s)))

## 4. Pairs and candidates

`geophys.compare(a, b)` says which of two videos violates physics, and
`geophys.pick([...])` returns the most plausible of several candidates. Both use the
paper's configurations, which combine four encoders (DINOv2, DINOv3, CORnet-S, VOneNet):
install CORnet-S and VOneNet, and accept the DINOv3 licence on Hugging Face, first.

In [ ]:
# !pip install -q git+https://github.com/dicarlolab/CORnet git+https://github.com/dicarlolab/vonenet
# print(geophys.compare(videos["plausible"][0], videos["teleport"][0]))
# print(geophys.pick([videos["teleport"][1], videos["plausible"][1], videos["vanish"][1]]))